# Folktale Classifier: Full Pipeline with BAAI/bge-m3
**Parts 1-3: Training, Confidence Testing, and Archetype Discovery**

Runs on GPU via Colab for fast embedding computation with the multilingual BAAI/bge-m3 model.

## Setup: Clone repo and install dependencies

In [ ]:
!pip install -q torch transformers sentence-transformers scikit-learn pandas numpy scipy pyyaml tqdm beautifulsoup4 requests matplotlib seaborn nltk

import os
os.chdir('/tmp')
!git clone https://github.com/cqiu-dot/asa-folktale-classifier.git 2>/dev/null || echo 'Repo may already exist'
os.chdir('/tmp/asa-folktale-classifier')

print("Setup complete!")

## Mount Google Drive (optional: if data is in Drive)

In [ ]:
# Uncomment if you need to access data from Google Drive
# from google.colab import drive
# drive.mount('/content/gdrive')
# !cp /content/gdrive/'My Drive'/aft.csv ../
# !cp /content/gdrive/'My Drive'/*.json ../

print("Drive mount ready (uncomment above if needed)")

## Update config to use bge-m3

In [ ]:
import yaml

# Load and update config
with open('config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Switch to bge-m3
config['model']['sentence_transformer'] = 'BAAI/bge-m3'
config['model']['embedding_dim'] = 1024

# Save updated config
with open('config/config.yaml', 'w') as f:
    yaml.dump(config, f)

print(f"Updated config:")
print(f"  Sentence Transformer: {config['model']['sentence_transformer']}")
print(f"  Embedding Dimension: {config['model']['embedding_dim']}")

## PART 1 & 2: Train Classifier and Test Confidence on Asian Tales

In [ ]:
import sys
sys.path.insert(0, '/tmp/asa-folktale-classifier')

import numpy as np
import pandas as pd
import json
import logging
from pathlib import Path
from sklearn.model_selection import train_test_split
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import FolktaleDataLoader
from src.model import FolktaleClassifier

logging.basicConfig(level=logging.WARNING)

print("="*80)
print("PART 1 & 2: Training and Confidence Testing")
print("="*80)

# Load Western tales
print("\nLoading Western folktales...")
western_df = pd.read_csv('../aft.csv')
western_df['text_processed'] = western_df['text'].fillna('').apply(lambda t: ' '.join(t.split()))

# Filter rare classes
label_counts = western_df['atu_id'].value_counts()
keep = label_counts[label_counts >= 3].index
western_df = western_df[western_df['atu_id'].isin(keep)].reset_index(drop=True)

texts = western_df['text_processed'].values
labels = western_df['atu_id'].values
print(f"  {len(texts)} tales, {len(set(labels))} ATU categories")

texts_train, texts_test, labels_train, labels_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)
texts_train, texts_val, labels_train, labels_val = train_test_split(
    texts_train, labels_train, test_size=0.1, random_state=42)

print(f"  Train: {len(texts_train)}, Val: {len(texts_val)}, Test: {len(texts_test)}")

# Train classifier
print("\nTraining classifier with bge-m3...")
clf = FolktaleClassifier(config, device='cuda')
metrics = clf.train(texts_train.tolist(), labels_train.tolist(),
                    texts_val.tolist(), labels_val.tolist())
print(f"  Train acc={metrics['train_accuracy']:.3f}  Val acc={metrics.get('val_accuracy', float('nan')):.3f}")

# Save classifier
Path('results/models').mkdir(parents=True, exist_ok=True)
clf.save('results/models/folktale_classifier.pkl')
print("  Classifier saved")

# Western test set confidence (baseline)
print("\nEvaluating on Western test set...")
_, probs_west = clf.predict(texts_test.tolist())
m_west = clf.get_confidence_metrics(probs_west)
mh_west = clf.mahalanobis_distances(texts_test.tolist())
K = probs_west.shape[1]

print(f"  K={K} classes, max entropy={np.log(K):.3f}")
print(f"  Western normalized entropy: {m_west['mean_normalized_entropy']:.4f}")
print(f"  Western Mahalanobis distance: {mh_west.mean():.4f}")

# Asian tales
print("\nLoading Asian/SE Asian folktales...")
loader = FolktaleDataLoader('config/config.yaml')
asian_df = loader.load_asian_tales()
print(f"  {len(asian_df)} tales ({asian_df['region'].value_counts().to_dict()})")

texts_asian = asian_df['text'].values.tolist()
_, probs_asian = clf.predict(texts_asian)
m_asian = clf.get_confidence_metrics(probs_asian)
mh_asian = clf.mahalanobis_distances(texts_asian)

print(f"  Asian normalized entropy: {m_asian['mean_normalized_entropy']:.4f}")
print(f"  Asian Mahalanobis distance: {mh_asian.mean():.4f}")

# Statistical comparison
def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na-1)*np.std(a,ddof=1)**2 + (nb-1)*np.std(b,ddof=1)**2) / (na+nb-2))
    return (np.mean(a) - np.mean(b)) / pooled

print("\n" + "="*80)
print("Hypothesis: ATU classifier is less confident on Asian tales")
print("="*80)

rows = [
    ("Max-softmax confidence", m_west['max_confidence'], m_asian['max_confidence']),
    ("Normalized entropy", m_west['normalized_entropy'], m_asian['normalized_entropy']),
    ("Margin p1-p2", m_west['margin'], m_asian['margin']),
    ("Mahalanobis distance", mh_west, mh_asian),
]

print(f"\n{'Metric':<30} {'Western':>9} {'Asian':>9} {'Delta':>9} {'Cohen d':>9}  {'MW p':>10}")
print("-"*90)
for label, w, a in rows:
    delta = np.mean(w) - np.mean(a)
    d = cohens_d(w, a)
    _, p = stats.mannwhitneyu(w, a, alternative='two-sided')
    print(f"{label:<30} {np.mean(w):>9.4f} {np.mean(a):>9.4f} {delta:>+9.4f} {d:>9.3f}  {p:>10.2e}")

# Save results
Path('results/analysis').mkdir(parents=True, exist_ok=True)
results = {
    'model': 'BAAI/bge-m3',
    'embedding_dim': 1024,
    'n_atu_classes': K,
    'western': {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in m_west.items()},
    'asian': {k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in m_asian.items()},
    'mahalanobis': {'western': mh_west.tolist(), 'asian': mh_asian.tolist()},
}
with open('results/analysis/confidence_results_bge.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\nPart 1 & 2 complete!")

## PART 3: Clustering and Archetype Discovery

In [ ]:
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression

print("\n" + "="*80)
print("PART 3: Clustering and Archetype Discovery")
print("="*80)

# Get embeddings for Asian tales
print("\nExtracting embeddings...")
embeddings = clf.encode_texts(texts_asian)
print(f"  Shape: {embeddings.shape}")

# K-means clustering
n_clusters = config['clustering']['kmeans']['n_clusters']
print(f"\nClustering with k={n_clusters}...")
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
cluster_assignments = kmeans.fit_predict(embeddings)
cluster_counts = np.bincount(cluster_assignments)
print(f"  Cluster sizes: min={cluster_counts.min()}, max={cluster_counts.max()}, mean={cluster_counts.mean():.1f}")

# Extract motifs
print("\nExtracting narrative motifs...")
from src.interpretability import MotifExtractor
motif_extractor = MotifExtractor(config)

top_tokens = motif_extractor.extract_top_tokens(embeddings, texts_asian, cluster_assignments)
top_ngrams = motif_extractor.extract_ngrams(texts_asian, cluster_assignments)
rep_stories = motif_extractor.extract_representative_stories(embeddings, texts_asian, asian_df, cluster_assignments)

# Logistic regression P(cluster | embedding)
print("\nTraining logistic regression P(cluster | embedding)...")
lr = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42)
lr.fit(embeddings, cluster_assignments)
print(f"  LR accuracy: {lr.score(embeddings, cluster_assignments):.3f}")

# Extract cluster features
cluster_features = {}
for cluster_id in range(n_clusters):
    coef = lr.coef_[cluster_id]
    top_indices = np.argsort(np.abs(coef))[-10:][::-1]
    cluster_features[cluster_id] = [(idx, float(coef[idx])) for idx in top_indices]

# Generate archetype proposals
print("\nGenerating archetype proposals...")
theme_keywords = {
    'royalty': ['king', 'queen', 'prince', 'princess', 'palace', 'court', 'emperor'],
    'magic': ['magic', 'spell', 'curse', 'wizard', 'enchant', 'supernatural'],
    'love': ['love', 'marry', 'bride', 'groom', 'romance', 'beloved'],
    'war': ['war', 'battle', 'soldier', 'enemy', 'sword', 'fight'],
    'animal': ['animal', 'beast', 'fox', 'tiger', 'monkey', 'bird'],
    'wisdom': ['wise', 'wisdom', 'teach', 'learn', 'advice', 'virtue'],
    'religion': ['buddha', 'monk', 'temple', 'prayer', 'enlighten', 'sacred'],
}

archetype_proposals = {}
for cluster_id in range(n_clusters):
    mask = cluster_assignments == cluster_id
    cluster_size = np.sum(mask)

    tokens = top_tokens[cluster_id][:5]
    ngrams = top_ngrams[cluster_id][:5]

    token_text = ' '.join([t[0].lower() for t in tokens])
    ngram_text = ' '.join([n[0].lower() for n in ngrams])
    combined_text = token_text + ' ' + ngram_text

    theme_scores = {}
    for theme, keywords in theme_keywords.items():
        score = sum(1 for kw in keywords if kw in combined_text)
        theme_scores[theme] = score

    top_theme = max(theme_scores, key=theme_scores.get)
    theme_label = {
        'royalty': 'Royal/Court Tales',
        'magic': 'Magical/Supernatural',
        'love': 'Romance/Love Stories',
        'war': 'War/Adventure Epics',
        'animal': 'Animal/Trickster Tales',
        'wisdom': 'Wisdom/Teaching Tales',
        'religion': 'Religious/Spiritual',
    }.get(top_theme, 'Narrative Cluster')

    archetype_proposals[cluster_id] = {
        'name': f"Cluster {cluster_id}: {theme_label}",
        'size': int(cluster_size),
        'theme': top_theme,
        'top_tokens': [(t[0], int(t[1])) for t in tokens],
        'top_ngrams': [(n[0], float(n[1])) for n in ngrams],
    }

# Display results
print("\n" + "="*80)
print("DISCOVERED NARRATIVE ARCHETYPES")
print("="*80)

for cluster_id in sorted(archetype_proposals.keys()):
    ap = archetype_proposals[cluster_id]
    print(f"\n[{ap['name']}]  (n={ap['size']})")
    print(f"  Theme: {ap['theme'].upper()}")
    print(f"  Top tokens: {', '.join([t[0] for t in ap['top_tokens']])}")
    print(f"  Key phrases: {', '.join([n[0] for n in ap['top_ngrams']])}")

# Save results
results_part3 = {
    'metadata': {
        'n_tales': len(texts_asian),
        'n_clusters': n_clusters,
        'embedding_dim': embeddings.shape[1],
        'embedding_model': 'BAAI/bge-m3',
        'clustering_method': 'kmeans',
    },
    'cluster_sizes': cluster_counts.tolist(),
    'archetypes': {
        str(k): v for k, v in archetype_proposals.items()
    },
    'logistic_regression_accuracy': float(lr.score(embeddings, cluster_assignments)),
}

with open('results/analysis/interpretability_results_bge.json', 'w') as f:
    json.dump(results_part3, f, indent=2)

np.save('results/analysis/asian_embeddings_bge.npy', embeddings)
np.save('results/analysis/cluster_assignments_bge.npy', cluster_assignments)

print("\n" + "="*80)
print("Part 3 complete! Results saved.")
print("="*80)

## Download Results

In [ ]:
from google.colab import files

# Create a zip file with all results
!cd /tmp/asa-folktale-classifier && zip -r results.zip results/ -q && mv results.zip /tmp/

# Download
files.download('/tmp/results.zip')
print("\nDownloaded: results.zip containing all analysis files")
print("Key files:")
print("  - results/analysis/confidence_results_bge.json (Parts 1 & 2 results)")
print("  - results/analysis/interpretability_results_bge.json (Part 3 archetypes)")
print("  - results/analysis/asian_embeddings_bge.npy (1024-dim embeddings)")
print("  - results/analysis/cluster_assignments_bge.npy (cluster assignments)")